In [28]:
import duckdb
import pandas as pd
import plotly.express as px


# 1. Verbindung zu DuckDB herstellen (In-Memory ist für Abfragen auf Parquet super schnell)
con = duckdb.connect(database=':memory:')

# 2. Den Pfad zu deinen Parquet-Dateien definieren (Passe den Pfad an dein Cluster an)
# Der Stern ** bedeutet: Suche in allen Unterordnern nach Parquet-Dateien
PARQUET_PATH = "/mnt/shared_data/finflow/mvbk_raw/*.parquet"

In [29]:
query_create_view = f"""
CREATE VIEW mvbk AS 
SELECT * FROM read_parquet('{PARQUET_PATH}', filename=true);
"""
con.execute(query_create_view)

In [30]:
query = f"""
SELECT *
FROM mvbk;
"""
con.execute(query).df()

,timestamp,location_long,location_lat,individual_local_identifier,ground_speed,heading,visible,filename
0,2016-06-27 18:00:00.000,NaN,NaN,Megan,NaN,NaN,True,/mnt/shared_data/finflow/mvbk_raw/study_101351...
1,2016-06-27 21:02:25.000,-90.557054,38.517922,Megan,0.12,0.0,True,/mnt/shared_data/finflow/mvbk_raw/study_101351...
2,2016-06-28 00:01:20.000,-90.557048,38.518124,Megan,0.07,0.0,True,/mnt/shared_data/finflow/mvbk_raw/study_101351...
3,2016-06-28 03:01:11.000,-90.556645,38.517865,Megan,0.06,0.0,True,/mnt/shared_data/finflow/mvbk_raw/study_101351...
4,2016-06-28 06:00:53.000,-90.556661,38.517912,Megan,0.23,0.0,True,/mnt/shared_data/finflow/mvbk_raw/study_101351...
...,...,...,...,...,...,...,...,...
24467,2015-08-02 12:35:45.000,-123.895400,37.100300,2015CA-Bph-05654,NaN,NaN,True,/mnt/shared_data/finflow/mvbk_raw/study_943824...
24468,2015-08-02 12:47:25.000,-123.901500,37.103600,2015CA-Bph-05654,NaN,NaN,True,/mnt/shared_data/finflow/mvbk_raw/study_943824...
24469,2015-08-02 12:59:26.000,-123.908200,37.106400,2015CA-Bph-05654,NaN,NaN,True,/mnt/shared_data/finflow/mvbk_raw/study_943824...
24470,2015-08-02 13:09:22.000,-123.914500,37.108700,2015CA-Bph-05654,NaN,NaN,True,/mnt/shared_data/finflow/mvbk_raw/study_943824...


In [31]:
import duckdb
import plotly.express as px

# WICHTIG: Passe den Ordnernamen hier exakt an den an, den Ray vorhin genutzt hat!
# In deinem vorherigen Log hieß er evtl. "mvbk_raw" statt "movebank_raw"
MOVEBANK_PATH = "/mnt/shared_data/finflow/mvbk_raw/*.parquet"

con = duckdb.connect(database=':memory:')

print("🔍 Analysiere die heruntergeladenen Rohdaten...")

# 1. Diagnose-Abfrage: Was haben wir da eigentlich genau heruntergeladen?
query_stats = f"""
SELECT 
    COUNT(*) as total_punkte,
    COUNT(DISTINCT individual_local_identifier) as anzahl_tiere,
    MIN(location_long) as min_lon, MAX(location_long) as max_lon
FROM read_parquet('{MOVEBANK_PATH}')
WHERE location_long IS NOT NULL
"""
df_stats = con.execute(query_stats).df()
print("\n📊 DATEN-STATISTIK:")
print(f"Punkte insgesamt: {df_stats['total_punkte'][0]:,}")
print(f"Anzahl verschiedener Tiere: {df_stats['anzahl_tiere'][0]}")
print(f"Längengrad-Spanne: {df_stats['min_lon'][0]} bis {df_stats['max_lon'][0]}\n")


# 2. Saubere Daten laden (Wir nutzen jetzt das 'visible'-Attribut von Movebank!)
# visible = true filtert automatisch alle von Forschern markierten Ausreißer weg!
query_clean = f"""
SELECT 
    location_long AS lon,
    location_lat AS lat,
    CAST(individual_local_identifier AS VARCHAR) AS tier_id
FROM read_parquet('{MOVEBANK_PATH}')
WHERE location_long IS NOT NULL 
  AND location_lat IS NOT NULL
  AND location_long != 0 
  AND location_lat != 0
  AND visible = true
"""

df_clean = con.execute(query_clean).df()
print(f"✅ {len(df_clean):,} saubere, verifizierte Punkte werden auf der Karte gezeichnet.")

if len(df_clean) > 0:
    fig = px.scatter_mapbox(
        df_clean, 
        lat='lat', 
        lon='lon', 
        color='tier_id',     
        zoom=3,              # Wir zoomen etwas näher heran
        mapbox_style="carto-darkmatter",
        title=f"FinFlow: Bereinigte Tierrouten ({len(df_clean):,} Punkte)",
        height=800
    )

    # Punkte noch kleiner und transparenter machen, damit Überlappungen sichtbar werden
    fig.update_traces(marker=dict(size=2, opacity=0.3))

    output_file = "finflow_movebank_bereinigt.html"
    fig.write_html(output_file)
    print(f"🗺️ Karte generiert! Bitte öffne '{output_file}'.")
else:
    print("❌ Nach der Bereinigung blieben keine Punkte übrig. (Falscher Pfad?)")

🔍 Analysiere die heruntergeladenen Rohdaten...

📊 DATEN-STATISTIK:
Punkte insgesamt: 24,016
Anzahl verschiedener Tiere: 41
Längengrad-Spanne: -131.4611 bis -90.2811876

✅ 23,996 saubere, verifizierte Punkte werden auf der Karte gezeichnet.


/tmp/ipykernel_4760/3711410388.py:47: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



🗺️ Karte generiert! Bitte öffne 'finflow_movebank_bereinigt.html'.
